# 诗歌生成

# 数据处理

In [11]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

start_token = 'bos'
end_token = 'eos'

'''
构建word2id	文本→数字，神经网络输入
构建id2word	数字→文本，解码输出
记录seqlen	支持动态填充或按长度批处理
'''
def process_dataset(fileName):
    examples = []
    with open(fileName, 'r',encoding='utf-8') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 按频次降序排序
    words, _ = zip(*sorted_counter)
    # *sorted_counter 解压成两个元组：所有词 和 所有频次
    # zip(*...) 将解压后的两个元组重新打包
    # words 接收第一个元组（词列表），_ 丢弃第二个元组（频次）
    
    words = ('PAD', 'UNK') + words[:len(words)] # 拼接结果：('PAD', 'UNK', 'c', 'a', 'b')
    
    word2id = dict(zip(words, range(len(words))))
    # zip(words, range(len(words)))：配对 (词, 索引)
    # 转换为字典：{'PAD':0, 'UNK':1, 'c':2, 'a':3, 'b':4}
    
    id2word = {word2id[k]:k for k in word2id}
    # 字典推导式，反转键值对：{0:'PAD', 1:'UNK', 2:'c', 3:'a', 4:'b'}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    # 嵌套列表推导：遍历每个 poem（一首诗），将其每个词 w 转换为对应的索引
    
    seqlen = [len(e) for e in indexed_examples] # 计算每个样本的原始长度（未填充前）
    
    instances = list(zip(indexed_examples, seqlen))
    # zip：将索引序列和长度配对
    # instances = [([2,3,4,6,7], 5), ([5,8,9,10], 4)]
    
    return instances, word2id, id2word

'''
构建诗词生成的训练数据集,用于语言模型训练（预测下一个词）
'''
def poem_dataset():
    instances, word2id, id2word = process_dataset('./poems.txt')
    '''
    instances：样本列表，每个元素是 (词索引序列, 原始长度)
    word2id：词到索引的映射字典
    id2word：索引到词的映射字典
    '''
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    # 将 Python 列表转换为 TensorFlow Dataset，每个样本格式为 (词序列, 长度)
    
    ds = ds.shuffle(buffer_size=10240)
    # 随机打乱样本顺序，从 10240 个样本中随机选取，防止过拟合
    
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    # 将样本按批次100打包，并对序列进行填充
    # 第一个元素（词序列）填充到批次内最大长度，第二个元素（长度标量）不填充
    # 批次: 词序列 = [[2,3,4,6,7], [5,8,9,10,0]]  # 填充到5
    # 长度 = [5, 4]  # 保持不变
    
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    # 将序列转换为 (输入, 目标) 对
    # 用途：语言模型训练 - 根据当前词预测下一个词
    '''
    原序列	     输入 (x[:, :-1])	目标 (x[:, 1:])	  新长度
    [2,3,4,6,7]	    [2,3,4,6]	      [3,4,6,7]	        4
    [5,8,9,10,0]	[5,8,9,10]	      [8,9,10,0]	    4
    '''
    return ds, word2id, id2word
    # ds：训练数据集，每个元素为 (输入序列, 目标序列, 新长度)

# 模型代码， 完成建模代码

In [23]:
'''
vocab_size（词表大小）:词汇表中不同词（或字符）的总数量
            决定输出层维度（每个位置预测7个词的概率）
            决定 Embedding 层的第一维（为每个词学习一个向量）
batch_size（批次大小）:一次训练或推理时，同时处理的样本数量
            训练：每次计算一个batch大小样本的平均梯度，参数更新更稳定
            推理：控制同时生成多少条文本
embedding_size（嵌入维度）：每个词被映射到的稠密向量维度
            将离散的词转换为连续的向量表示
'''

class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.rnncell = tf.keras.layers.SimpleRNNCell(128)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)

    # 用于训练，输入完整序列 (batch, seq_len)，输出所有位置logits
    @tf.function
    def call(self, inp_ids):
        '''
        此处完成建模过程，可以参考Learn2Carry
        输入: inp_ids shape (batch_size, seq_len)
        输出: logits shape (batch_size, seq_len, vocab_size)
        '''
        # 1. Embedding：将词索引转换为向量
        emb=self.embed_layer(inp_ids)    # (batch, seq_len, 64)
        # 2. RNN层：处理整个序列
        rnn_out=self.rnn_layer(emb)      # (batch, seq_len, 128)
        # 3. 全连接层：映射到词表大小
        logits=self.dense(rnn_out)       # (batch, seq_len, v_sz)
        
        return logits

    # 用于推理/生成，输入——单步输入 + 状态，输出——下一个词 + 新状态
    @tf.function
    def get_next_token(self, x, state):
        '''
        x: 当前输入词索引, shape(x) = [b_sz,] 
        state: 上一时刻的RNN状态，shape (batch_size, 128)
        输出:
            out: 预测的下一个词索引，shape (batch_size,)
            state: 当前时刻的新状态
        '''

        # print(f"x shape: {x.shape}")
        # print(f"state shape: {state.shape if hasattr(state, 'shape') else 'list'}")
    
        # 1. Embedding：将当前词转为向量
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        # print(f"inp_emb shape: {inp_emb.shape}")
        
        # 2. RNNCell：单步计算
        # 将inp_emb增加时间维（从(batch,64)变为(batch,1,64)）
        # inp_emb = tf.expand_dims(inp_emb, axis=1)  # (batch, 1, 64)
        # print(f"inp_emb after expand: {inp_emb.shape}")
        h, state = self.rnncell.call(inp_emb, state) # shape(b_sz, h_sz)
        # print(f"h shape: {h.shape}")
        # print(f"state shape after: {state.shape}")

        # 3. 全连接层：得到logits
        logits = self.dense(h) # shape(b_sz, v_sz)
        # print(f"logits shape: {logits.shape}")
        
        # 4. 取概率最大的词
        out = tf.argmax(logits, axis=-1)  # (batch,)
        # return out, state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [19]:
'''
可以对任意维度进行平均（不仅是序列维度）
自动处理维度不匹配，通过广播实现
'''

# 创建掩码矩阵 [batch_size, maxLen]
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])  # [batch_size, maxLen]

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)

# 掩码平均
def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : 待平均的张量 shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : 要平均的维度（整数）which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()

    # 确保 lengths 的维度和要平均的维度索引匹配
    # 将 lengths 和 mask 扩展维度，使其能与 reduce_target 进行广播运算
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
        # % (dim, len(...)) 将两个值依次填入 %d
        
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    '''
    reduce_target 的秩 = A
    lengths 的秩 = B
    dim 是要平均的维度
    
    约束：B = dim（由前面检查保证）
    
    所以：
    rank_diff = A - B - 1
             = A - dim - 1    
    '''
    # 计算秩差和形状
    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)

    # 形状对齐，将长度和掩码扩展到与 reduce_target 相同的维度，以便广播
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)

    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)
    # lengths_reshape: (batch_size, 1)  # 增加维度用于广播
    # mask: (batch_size, seq_len, 1)    # 增加最后一维

    # 应用掩码并平均
    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [20]:
@tf.function
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

@tf.function(reduce_retracing=True)
def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步优化过程，可以参考之前做过的模型
    '''
    with tf.GradientTape() as tape:
        logits=model(x)  # 前向传播，输出 (batch, seq_len, vocab_size)
        loss=compute_loss(logits,y,seqlen)  # 计算损失（已考虑掩码）

    # 计算梯度
    grads=tape.gradient(loss,model.trainable_variables)
    # 更新参数
    optimizer.apply_gradients(zip(grads,model.trainable_variables))
    
    return loss

def train(epoch, model, optimizer, ds):
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [21]:
optimizer = optimizers.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

for epoch in range(10):
    loss = train(epoch, model, optimizer, train_ds)

epoch 0 : loss 8.8204975
epoch 1 : loss 6.582925
epoch 2 : loss 6.137913
epoch 3 : loss 5.834094
epoch 4 : loss 5.736978
epoch 5 : loss 5.5486073
epoch 6 : loss 5.3733783
epoch 7 : loss 5.371621
epoch 8 : loss 5.2664227
epoch 9 : loss 5.280746


# 生成过程

In [30]:
def gen_sentence(begin_word='日'):
    if begin_word not in word2id:
        print(f"警告：'{begin_word}' 不在词表中，使用 'bos' 代替")
        begin_word='bos'
    state = tf.random.normal(shape=(1, 128), stddev=0.5)  # 单个张量
    cur_token = tf.constant([word2id[begin_word]], dtype=tf.int32)
    collect = [begin_word]
    for _ in range(50):
        cur_token, state = model.get_next_token(cur_token, state)
        # next_word=id2word[cur_token.numpy()[0]]
        # if next_word=='eos':
        #     break
        # collect.append(next_word)
        collect.append(id2word[cur_token.numpy()[0]])
    # return [id2word[t] for t in collect]
    return ''.join(collect)
# print(''.join(gen_sentence()))
print("以'日'开头：", gen_sentence('日'))
print("以'红'开头：", gen_sentence('红'))
print("以'湖'开头：", gen_sentence('湖'))
print("以'夜'开头：", gen_sentence('夜'))
print("以'月'开头：", gen_sentence('月'))
print("以'山'开头：", gen_sentence('山'))
print("以'海'开头：", gen_sentence('海'))

以'日'开头： 日日夜来无处，春风吹落花。eos来无限处，不见此人心。eos有人间事，无人不得人。eos来无处处，不见此人心。eos有
以'红'开头： 红蓉柳枝红叶。eos来无处处，风雨满花飞。eos月无人去，春风月上春。eos来无处处，不见此人心。eos有人间事，无人
以'湖'开头： 湖，一得无人不可知。eos道不知何处处，不知何处是何人。eos来不得人相见，不得人间不得人。eos道不知人不见，不
以'夜'开头： 夜闻君人到处，不见人间。eos来何处处，不见人间人。eos有无人事，何人不可知。eos来无处处，不见此人心。eos有人
以'月'开头： 月中来，此日无人事。eos来不可知，不见人中去。eos有无人间，不知何所见。eos有无人间，不见无人间。eos有无人事
以'山'开头： 山畔里相逢，此人无人。eos人不见，何事不知。eos有无人，不知何所。eos之不见，何以何人。eos人不见，何事不知。
以'海'开头： 海上山，一年无处处。eos来不得人相见，不得人间不得人。eos道不知人不见，不知何处是何人。eos来不得人相见，不
